In [ ]:
# ============================================================
# RepoSage: Multi-Agent AI Codebase Auditor & Modernization Tool
# Complete Google Colab Single-Cell Implementation
# ============================================================

%pip -q install -U langgraph langchain-core langchain-google-genai gradio

import os
import zipfile
import tempfile
import shutil
from pathlib import Path
from getpass import getpass
from typing import TypedDict
from IPython.display import display, Markdown

from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
import gradio as gr

# -------------------- API KEY --------------------
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass(
        "Enter your Gemini API key (input hidden): "
    )

MODEL_NAME = "gemini-3.5-flash-lite"

llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=0.2
)

# -------------------- REPOSITORY READER --------------------
ALLOWED_EXTENSIONS = {
    ".py", ".js", ".jsx", ".ts", ".tsx", ".java",
    ".c", ".cpp", ".h", ".hpp", ".html", ".css",
    ".sql", ".md", ".json", ".yaml", ".yml",
    ".xml", ".toml", ".ini"
}

IGNORED_DIRS = {
    ".git", "node_modules", "venv", ".venv",
    "__pycache__", "dist", "build", ".next"
}

CONFIG_FILES = {
    "requirements.txt", "package.json", "pom.xml",
    "build.gradle", "Dockerfile", "Makefile",
    ".gitignore"
}

def extract_repository(zip_path):
    """Read source files from a ZIP without executing them."""
    temp_dir = tempfile.mkdtemp(prefix="reposage_")

    try:
        with zipfile.ZipFile(zip_path, "r") as archive:
            root = Path(temp_dir).resolve()
            for member in archive.infolist():
                target = (root / member.filename).resolve()
                if target != root and root not in target.parents:
                    raise ValueError("Unsafe path found in ZIP archive.")
            archive.extractall(root)

        files = []
        count = 0

        for file in root.rglob("*"):
            if not file.is_file():
                continue

            if any(part in IGNORED_DIRS for part in file.parts):
                continue

            if file.suffix.lower() not in ALLOWED_EXTENSIONS \
                    and file.name not in CONFIG_FILES:
                continue

            if file.name.lower() in {
                ".env", "id_rsa", "id_ed25519",
                "credentials.json", "secrets.json"
            }:
                continue

            try:
                content = file.read_text(
                    encoding="utf-8", errors="ignore"
                )
            except Exception:
                continue

            content = content[:8000]
            relative_name = file.relative_to(root)

            files.append(
                f"\n--- FILE: {relative_name} ---\n{content}"
            )
            count += 1

            if count >= 50:
                files.append("\n[File limit reached for this demo.]")
                break

        if not files:
            raise ValueError(
                "No supported source files found. Upload a ZIP "
                "containing Python, JavaScript, Java or other supported files."
            )

        return "\n".join(files)[:65000]

    finally:
        shutil.rmtree(temp_dir, ignore_errors=True)


# -------------------- LANGGRAPH STATE --------------------
class RepoState(TypedDict, total=False):
    repo_text: str
    code_analysis: str
    security_analysis: str
    architecture_analysis: str
    dependency_analysis: str
    testing_analysis: str
    performance_analysis: str
    review: str
    final_report: str


# -------------------- LLM HELPER --------------------
def ask_agent(role, instructions, repo_text=None, reports=None):
    prompt = f"""
You are the {role} in RepoSage, a software analysis system.

Rules:
- Use only evidence available in the supplied material.
- Do not invent filenames, vulnerabilities, dependencies or test results.
- Clearly distinguish observed evidence from possible risks.
- If evidence is insufficient, say "Needs verification".
- Give concise, structured, actionable results.
- This is a static review. You have not executed the code.

TASK:
{instructions}

"""
    if repo_text is not None:
        prompt += "\nREPOSITORY SOURCE:\n" + repo_text

    if reports is not None:
        prompt += "\nAGENT REPORTS:\n" + reports

    response = llm.invoke([HumanMessage(content=prompt)])
    content = response.content

    if isinstance(content, list):
        return "\n".join(
            str(item.get("text", item))
            if isinstance(item, dict) else str(item)
            for item in content
        )

    return str(content)


# -------------------- AGENT 1: CODE UNDERSTANDING --------------------
def code_agent(state):
    result = ask_agent(
        "Code Understanding Agent",
        """
Analyze the project languages, important files, functions, classes,
code organization, maintainability, duplication, error handling and
code quality. Give an estimated code-quality score out of 100 and
explain that it is an AI estimate, not a measured benchmark.
Cite filenames where possible.
""",
        repo_text=state["repo_text"]
    )
    return {"code_analysis": result}


# -------------------- AGENT 2: SECURITY --------------------
def security_agent(state):
    result = ask_agent(
        "Security Audit Agent",
        """
Perform a defensive static security review. Look for possible
injection risks, unsafe input handling, authentication weaknesses,
hard-coded credentials visible in the supplied files, unsafe file
operations and insecure configuration. Assign severity only when
evidence supports it. Give evidence, impact and remediation.
Do not claim that a vulnerability is confirmed without sufficient evidence.
""",
        repo_text=state["repo_text"]
    )
    return {"security_analysis": result}


# -------------------- AGENT 3: ARCHITECTURE --------------------
def architecture_agent(state):
    result = ask_agent(
        "Architecture Agent",
        """
Infer the likely architecture from the supplied source. Explain
major components, layers, data flow, coupling and maintainability.
Suggest a practical target architecture appropriate to the project's size.
Do not claim to have discovered components that are not shown.
""",
        repo_text=state["repo_text"]
    )
    return {"architecture_analysis": result}


# -------------------- AGENT 4: DEPENDENCIES --------------------
def dependency_agent(state):
    result = ask_agent(
        "Dependency Agent",
        """
Inspect dependency and configuration files. Identify visible
version-pinning issues, dependency-management gaps and upgrade priorities.
Do not claim a package is currently vulnerable or outdated unless
the supplied evidence establishes that. Explain how to verify concerns.
""",
        repo_text=state["repo_text"]
    )
    return {"dependency_analysis": result}


# -------------------- AGENT 5: TESTING --------------------
def testing_agent(state):
    result = ask_agent(
        "Testing Agent",
        """
Review visible test coverage and suggest prioritized test cases.
Include unit, integration, edge-case and security tests where appropriate.
For each suggested test, explain what it should verify.
Do not claim tests have been run.
""",
        repo_text=state["repo_text"]
    )
    return {"testing_analysis": result}


# -------------------- AGENT 6: PERFORMANCE --------------------
def performance_agent(state):
    result = ask_agent(
        "Performance Optimization Agent",
        """
Analyze the codebase for computational efficiency, memory usage inefficiencies,
bottlenecks, unoptimized storage patterns, database indexing opportunities, and resource
allocation flaws. Highlight findings with evidence and present actionable optimization strategies.
""",
        repo_text=state["repo_text"]
    )
    return {"performance_analysis": result}


# -------------------- AGENT 7: CRITIC / REVIEWER --------------------
def reviewer_agent(state):
    reports = f"""
CODE ANALYSIS:
{state.get('code_analysis', '')}

SECURITY ANALYSIS:
{state.get('security_analysis', '')}

ARCHITECTURE ANALYSIS:
{state.get('architecture_analysis', '')}

DEPENDENCY ANALYSIS:
{state.get('dependency_analysis', '')}

TESTING ANALYSIS:
{state.get('testing_analysis', '')}

PERFORMANCE ANALYSIS:
{state.get('performance_analysis', '')}
"""
    result = ask_agent(
        "Independent Reviewer Agent",
        """
Critically review the reports. Identify contradictions, unsupported
claims, duplicated findings and missing evidence. Separate confirmed
observations from hypotheses. Prioritize the three most important
findings and explain what a human reviewer should verify.
""",
        reports=reports
    )
    return {"review": result}


# -------------------- AGENT 8: SUPERVISOR / FINAL REPORT --------------------
def final_agent(state):
    reports = f"""
CODE:
{state.get('code_analysis', '')}

SECURITY:
{state.get('security_analysis', '')}

ARCHITECTURE:
{state.get('architecture_analysis', '')}

DEPENDENCIES:
{state.get('dependency_analysis', '')}

TESTING:
{state.get('testing_analysis', '')}

PERFORMANCE:
{state.get('performance_analysis', '')}

INDEPENDENT REVIEW:
{state.get('review', '')}
"""
    result = ask_agent(
        "Supervisor and Final Report Agent",
        """
Create a professional Markdown report containing:

1. Project overview
2. Technologies and repository structure
3. Overall code health assessment
4. Code quality findings
5. Security findings, evidence and severity
6. Architecture assessment
7. Dependency management review
8. Testing strategy
9. Performance Optimization Assessment (Include detailed efficiency, memory, and database opportunities)
10. Top ten prioritized improvements
11. A practical 30/60/90-day modernization roadmap
12. Limitations and items requiring human verification

Do not invent facts. If a score is included, label it an AI estimate.
Finish with a concise executive summary.
""",
        reports=reports
    )
    return {"final_report": result}


# -------------------- BUILD LANGGRAPH --------------------
workflow = StateGraph(RepoState)

workflow.add_node("code", code_agent)
workflow.add_node("security", security_agent)
workflow.add_node("architecture", architecture_agent)
workflow.add_node("dependency", dependency_agent)
workflow.add_node("testing", testing_agent)
workflow.add_node("performance", performance_agent)
workflow.add_node("reviewer", reviewer_agent)
workflow.add_node("final", final_agent)

# Specialized agents run independently before the reviewer.
workflow.add_edge(START, "code")
workflow.add_edge(START, "security")
workflow.add_edge(START, "architecture")
workflow.add_edge(START, "dependency")
workflow.add_edge(START, "testing")
workflow.add_edge(START, "performance")

workflow.add_edge(
    ["code", "security", "architecture", "dependency", "testing", "performance"],
    "reviewer"
)

workflow.add_edge("reviewer", "final")
workflow.add_edge("final", END)

app = workflow.compile()


# -------------------- RUN THE MULTI-AGENT SYSTEM --------------------
def analyze_repository(uploaded_file):
    if not uploaded_file:
        return (
            "Please upload a project ZIP file.",
            "No analysis performed.",
            None
        )

    try:
        repo_text = extract_repository(uploaded_file)

        result = app.invoke({"repo_text": repo_text})

        report = result["final_report"]

        trace = """
### Agent Execution Trace

1. Code Understanding Agent
2. Security Audit Agent
3. Architecture Agent
4. Dependency Agent
5. Testing Agent
6. Performance Optimization Agent
7. Independent Reviewer Agent
8. Supervisor / Final Report Agent

**Workflow completed.** Each report is AI-generated and requires
verification before acting on security findings.
"""

        report_path = Path("/content/RepoSage_Final_Report.md")
        report_path.write_text(report, encoding="utf-8")

        return report, trace, str(report_path)

    except Exception as error:
        return (
            f"### Analysis Error\n\n`{type(error).__name__}: {error}`\n\n"
            "Check your API key, model availability, internet connection "
            "and uploaded ZIP file, then try again.",
            "Workflow did not complete.",
            None
        )


# -------------------- GRADIO USER INTERFACE --------------------
with gr.Blocks(
    title="RepoSage - AI Codebase Auditor"
) as demo:

    gr.Markdown("""
# 🧠 RepoSage
## Autonomous Multi-Agent AI Codebase Auditor & Modernization Agent

Upload a ZIP of a software project. Specialized AI agents will inspect
the source and produce an evidence-based report.

**Capabilities:** Code review · Security review · Architecture analysis ·
Dependency review · Test planning · Performance optimization · Independent critique · Modernization roadmap

**Privacy:** Use a sample project. Do not upload private code, credentials,
API keys or sensitive information.
""")

    with gr.Row():
        file_input = gr.File(
            label="Upload source-code ZIP",
            file_types=[".zip"],
            type="filepath"
        )

    analyze_button = gr.Button(
        "🚀 Analyze Repository",
        variant="primary"
    )

    execution_trace = gr.Markdown(label="Agent Workflow")
    output_report = gr.Markdown(label="Final Audit Report")

    report_download = gr.File(
        label="Download Final Markdown Report"
    )

    analyze_button.click(
        fn=analyze_repository,
        inputs=[file_input],
        outputs=[
            output_report,
            execution_trace,
            report_download
        ]
    )

print("RepoSage initialized. Open the Gradio interface below.")
demo.launch(share=True, debug=True, theme=gr.themes.Soft())


RepoSage initialized. Open the Gradio interface below.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b5c30666b25eff59bd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_